## Задание 3

In [1]:
import pandas as pd

df = pd.read_csv('type_issues.csv')

### Задание 1. Определение текущих типов

In [2]:
df.dtypes  # типы всех столбцов

TransactionID          str
Date                   str
ProductCategory        str
SalesAmount            str
Quantity           float64
Discount           float64
IsPromo                str
MixedColumn            str
dtype: object

In [3]:
print(f"Память датафрейма: {df.memory_usage(deep=True).sum() / 1024:.1f} КБ")
df.info(memory_usage='deep')

Память датафрейма: 50.1 КБ
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    150 non-null    str    
 1   Date             150 non-null    str    
 2   ProductCategory  150 non-null    str    
 3   SalesAmount      147 non-null    str    
 4   Quantity         150 non-null    float64
 5   Discount         150 non-null    float64
 6   IsPromo          150 non-null    str    
 7   MixedColumn      128 non-null    str    
dtypes: float64(2), str(6)
memory usage: 50.1 KB


In [4]:
df.head(10)  # проблемные места видны при загрузке

,TransactionID,Date,ProductCategory,SalesAmount,Quantity,Discount,IsPromo,MixedColumn
0,TX0001,02/03/2026,Clothing,76.39,3.0,0.06,No,8
1,TX0002,07/01/2026,Clothing,445.05,3.0,0.18,No,unknown
2,TX0003,23.01.2026,HOME,$466.98,5.0,0.08,Yes,unknown
3,TX0004,03-18-2026,Toys,$228.33,4.0,0.28,Yes,unknown
4,TX0005,25/01/2026,BOOKS,200.24,3.0,0.06,Yes,8
5,TX0006,2026-01-11,Home,$136.22,4.0,0.23,No,8
6,TX0007,09.02.2026,Toys,$332.11,1.0,0.22,No,NaN
7,TX0008,03-26-2026,Clothing,251.38,5.0,0.26,No,8
8,TX0009,21/01/2026,Toys,482.48,2.0,0.16,No,8
9,TX0010,03-16-2026,Home,402.48,4.0,0.06,No,NaN


Проблемы с типами:
- `SalesAmount` — объектный тип (строка): значения с `$`, пробелами и пустые ячейки;
- `Quantity` — `float64` вместо целого (значения вида `2.0`);
- `Date` — строка: смесь форматов (`DD/MM/YYYY`, `DD.MM.YYYY`, `MM-DD-YYYY`, `YYYY-MM-DD`) и текст `'not a date'`;
- `IsPromo` — строка с вариантами `Yes`/`No` в разном регистре;
- `ProductCategory` — строка: лишние пробелы и разнобой регистра.
- `Discount` — уже корректный `float64`, `TransactionID` — строка (как и должно быть).

### Задание 2. Приведение числовых столбцов

#### `SalesAmount` → `float` (убираем `$` и пробелы)

In [5]:
df['SalesAmount'] = df['SalesAmount'].str.replace('$', '', regex=False)
df['SalesAmount'] = df['SalesAmount'].str.strip()
df['SalesAmount'].head(10)

0     76.39
1    445.05
2    466.98
3    228.33
4    200.24
5    136.22
6    332.11
7    251.38
8    482.48
9    402.48
Name: SalesAmount, dtype: str

In [6]:
df['SalesAmount'] = pd.to_numeric(df['SalesAmount'], errors='coerce')  # ошибки -> NaN
print(f"SalesAmount: тип {df['SalesAmount'].dtype}, пропусков: {df['SalesAmount'].isna().sum()}")
df['SalesAmount'].head(10)

SalesAmount: тип float64, пропусков: 3


0     76.39
1    445.05
2    466.98
3    228.33
4    200.24
5    136.22
6    332.11
7    251.38
8    482.48
9    402.48
Name: SalesAmount, dtype: float64

#### `Quantity` → `int` (через `pd.to_numeric` + `astype('Int64')`)

In [7]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')
print(f"Quantity: тип {df['Quantity'].dtype}")
df['Quantity'].head(10)

Quantity: тип Int64


0    3
1    3
2    5
3    4
4    3
5    4
6    1
7    5
8    2
9    4
Name: Quantity, dtype: Int64

### Задание 3. Приведение дат

In [8]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True, errors='coerce')
print(f"Корректных дат: {df['Date'].notna().sum()} из {len(df)}  (NaT: {df['Date'].isna().sum()})")
df['Date'].head(10)

Корректных дат: 144 из 150  (NaT: 6)


0   2026-03-02
1   2026-01-07
2   2026-01-23
3   2026-03-18
4   2026-01-25
5   2026-11-01
6   2026-02-09
7   2026-03-26
8   2026-01-21
9   2026-03-16
Name: Date, dtype: datetime64[us]

In [9]:
df[df['Date'].isna()]  # строки, где вместо даты текст 'not a date' -> NaT

,TransactionID,Date,ProductCategory,SalesAmount,Quantity,Discount,IsPromo,MixedColumn
10,TX0011,NaT,Clothing,233.05,4,0.12,Yes,unknown
11,TX0012,NaT,TOYS,402.52,5,0.18,No,8
12,TX0013,NaT,ELECTRONICS,30.44,2,0.14,Yes,8
13,TX0014,NaT,electronics,387.03,2,0.05,Yes,unknown
14,TX0015,NaT,BOOKS,11.55,3,0.11,Yes,NaN
15,TX0016,NaT,Clothing,153.48,5,0.26,Yes,8


### Задание 4. Категориальные данные

#### Очистка: убираем пробелы, приводим к нижнему регистру

In [10]:
mem_before = df['ProductCategory'].memory_usage(deep=True)
df['ProductCategory'] = df['ProductCategory'].str.strip().str.lower()
df['ProductCategory'].unique()

<StringArray>
['clothing', 'home', 'toys', 'books', 'electronics']
Length: 5, dtype: str

#### Преобразование в `category` и сравнение памяти

In [11]:
df['ProductCategory'] = df['ProductCategory'].astype('category')
mem_after = df['ProductCategory'].memory_usage(deep=True)
print(f"Память до: {mem_before} байт, после: {mem_after} байт")
print(f"Экономия памяти: {100 * (1 - mem_after / mem_before):.1f}%")
df['ProductCategory']

Память до: 8529 байт, после: 559 байт
Экономия памяти: 93.4%


0         clothing
1         clothing
2             home
3             toys
4            books
          ...     
145          books
146    electronics
147          books
148    electronics
149    electronics
Name: ProductCategory, Length: 150, dtype: category
Categories (5, str): ['books', 'clothing', 'electronics', 'home', 'toys']

### Задание 5. Логические значения

#### `IsPromo` → `bool` через маппинг

In [12]:
promo_map = {'Yes': True, 'No': False, 'yes': True, 'no': False}
df['IsPromo'] = df['IsPromo'].map(promo_map).astype('bool')
print(f"Тип: {df['IsPromo'].dtype}")
df['IsPromo'].value_counts()

Тип: bool


IsPromo
False    88
True     62
Name: count, dtype: int64

### Итоговые типы

In [13]:
df.dtypes

TransactionID                 str
Date               datetime64[us]
ProductCategory          category
SalesAmount               float64
Quantity                    Int64
Discount                  float64
IsPromo                      bool
MixedColumn                   str
dtype: object

In [14]:
df.to_csv('clean_type_issues.csv', index=False)
print(f"Сохранено строк: {df.shape[0]}")
df.head()

Сохранено строк: 150


,TransactionID,Date,ProductCategory,SalesAmount,Quantity,Discount,IsPromo,MixedColumn
0,TX0001,2026-03-02,clothing,76.39,3,0.06,False,8
1,TX0002,2026-01-07,clothing,445.05,3,0.18,False,unknown
2,TX0003,2026-01-23,home,466.98,5,0.08,True,unknown
3,TX0004,2026-03-18,toys,228.33,4,0.28,True,unknown
4,TX0005,2026-01-25,books,200.24,3,0.06,True,8
